# BGR Node Classification with TopologicPy

This notebook classifies nodes in one Building Graph Representation (BGR):

- `0`: ground
- `1`: column
- `3`: office
- `4`: core

It first repairs the exported node labels using the existing one-hot columns,
then trains a GraphSAGE node-classification model using coordinates and graph
connectivity. The one-hot category columns are removed from the model inputs to
avoid leaking the answers into the features.


## 1. Imports and environment

In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

required_packages = {
    "topologicpy": "topologicpy==0.9.43",
    "torch": "torch",
    "torch-geometric": "torch-geometric",
    "scikit-learn": "scikit-learn",
    "PyYAML": "pyyaml",
}
for package_name, install_name in required_packages.items():
    try:
        version(package_name)
    except Exception:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", install_name]
        )

import numpy as np
import pandas as pd
import torch
import yaml
from topologicpy.Helper import Helper
from topologicpy.PyG import PyG

print("Python:", sys.executable)
print("TopologicPy:", version("topologicpy"))
print("PyTorch:", torch.__version__)


c:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: c:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\.gmlenv\Scripts\python.exe
TopologicPy: 0.9.43
PyTorch: 2.12.0+cpu


## 2. Locate the exported BGR dataset

In [2]:
source_candidates = [
    Path("..") / "bgr_dataset",
    Path("assignment03") / "bgr_dataset",
    Path("bgr_dataset"),
]
SOURCE_DATASET = next(
    (path.resolve() for path in source_candidates if path.is_dir()),
    None,
)
if SOURCE_DATASET is None:
    raise FileNotFoundError("Could not find assignment03/bgr_dataset")

required_files = ["graphs.csv", "nodes.csv", "edges.csv"]
missing = [
    name for name in required_files
    if not (SOURCE_DATASET / name).is_file()
]
if missing:
    raise FileNotFoundError(
        f"Missing files in {SOURCE_DATASET}: {missing}"
    )

NODE_DATASET = SOURCE_DATASET.parent / "bgr_node_dataset"
MODEL_PATH = SOURCE_DATASET.parent / "bgr_node_model.pt"

print("Source:", SOURCE_DATASET)
print("Node dataset:", NODE_DATASET)
print("Model:", MODEL_PATH)


Source: C:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\assignment03\bgr_dataset
Node dataset: C:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\assignment03\bgr_node_dataset
Model: C:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\assignment03\bgr_node_model.pt


## 3. Generate the node-classification CSV dataset

In [3]:
nodes = pd.read_csv(SOURCE_DATASET / "nodes.csv")
edges = pd.read_csv(SOURCE_DATASET / "edges.csv")
graphs = pd.read_csv(SOURCE_DATASET / "graphs.csv")

one_hot_columns = [
    f"feat_feature_{index:02d}" for index in range(5)
]
missing_features = [
    column for column in one_hot_columns
    if column not in nodes.columns
]
if missing_features:
    raise ValueError(
        f"Cannot derive node labels; missing columns: {missing_features}"
    )

# Recover the original cell_type from the exported one-hot vector.
nodes["cell_type"] = (
    nodes[one_hot_columns]
    .to_numpy()
    .argmax(axis=1)
    .astype(int)
)

# PyTorch classification targets must be contiguous: 0, 1, 2, 3.
cell_type_to_label = {0: 0, 1: 1, 3: 2, 4: 3}
label_to_name = {0: "ground", 1: "column", 2: "office", 3: "core"}
nodes["label"] = nodes["cell_type"].map(cell_type_to_label)
if nodes["label"].isna().any():
    unknown = sorted(nodes.loc[nodes["label"].isna(), "cell_type"].unique())
    raise ValueError(f"Unknown cell types: {unknown}")
nodes["label"] = nodes["label"].astype(int)

# Use geometry as model input. The graph edges provide neighbourhood context.
nodes["feat_x"] = nodes["X"].astype(float)
nodes["feat_y"] = nodes["Y"].astype(float)
nodes["feat_z"] = nodes["Z"].astype(float)

for column in ["feat_x", "feat_y", "feat_z"]:
    standard_deviation = nodes[column].std()
    if standard_deviation > 0:
        nodes[column] = (
            nodes[column] - nodes[column].mean()
        ) / standard_deviation

# Create deterministic stratified masks. Classes represented by only one node
# remain in training because they cannot be split across all three sets.
rng = np.random.default_rng(42)
nodes["train_mask"] = False
nodes["val_mask"] = False
nodes["test_mask"] = False

for label, group in nodes.groupby("label"):
    indices = group.index.to_numpy(copy=True)
    rng.shuffle(indices)
    count = len(indices)

    if count < 3:
        train_indices = indices
        val_indices = np.array([], dtype=int)
        test_indices = np.array([], dtype=int)
    else:
        val_count = max(1, round(count * 0.15))
        test_count = max(1, round(count * 0.15))
        train_count = count - val_count - test_count
        train_indices = indices[:train_count]
        val_indices = indices[train_count:train_count + val_count]
        test_indices = indices[train_count + val_count:]

    nodes.loc[train_indices, "train_mask"] = True
    nodes.loc[val_indices, "val_mask"] = True
    nodes.loc[test_indices, "test_mask"] = True

# Remove the one-hot columns because they directly encode the target label.
nodes = nodes.drop(columns=one_hot_columns)

NODE_DATASET.mkdir(parents=True, exist_ok=True)
nodes.to_csv(NODE_DATASET / "nodes.csv", index=False)
edges.to_csv(NODE_DATASET / "edges.csv", index=False)
graphs.to_csv(NODE_DATASET / "graphs.csv", index=False)

meta_source = SOURCE_DATASET / "meta.yaml"
if meta_source.exists():
    shutil.copy2(meta_source, NODE_DATASET / "meta.yaml")

print("Training label counts:")
print(nodes["label"].value_counts().sort_index())
print("\nOriginal cell-type counts:")
print(nodes["cell_type"].value_counts().sort_index())
print("\nLabel names:", label_to_name)
print("\nMask counts:")
print(nodes[["train_mask", "val_mask", "test_mask"]].sum())
print("\nFeature columns:")
print([column for column in nodes.columns if column.startswith("feat_")])


Training label counts:
label
0     1
1    36
2    17
3     1
Name: count, dtype: int64

Original cell-type counts:
cell_type
0     1
1    36
3    17
4     1
Name: count, dtype: int64

Label names: {0: 'ground', 1: 'column', 2: 'office', 3: 'core'}

Mask counts:
train_mask    39
val_mask       8
test_mask      8
dtype: int64

Feature columns:
['feat_x', 'feat_y', 'feat_z']


## 4. Load the node-classification dataset

In [4]:
pyg = PyG.ByCSVPath(
    path=str(NODE_DATASET),
    level="node",
    task="classification",
    nodeLabelType="categorical",
)

print(pyg.Summary())


{'level': 'node', 'task': 'classification', 'graph_label_type': 'categorical', 'node_label_type': 'categorical', 'edge_label_type': 'categorical', 'cv': 'holdout', 'split': (0.8, 0.1, 0.1), 'k_folds': 5, 'holdout_group_by': None, 'conv': 'sage', 'hidden_dims': (64, 64), 'activation': 'relu', 'dropout': 0.1, 'batch_norm': False, 'residual': False, 'pooling': 'mean', 'epochs': 50, 'batch_size': 32, 'lr': 0.001, 'weight_decay': 0.0, 'optimizer': 'adam', 'gradient_clip_norm': None, 'early_stopping': False, 'early_stopping_patience': 10, 'device': 'cpu', 'ontology': True, 'ontology_metadata_columns': {'graph': ['label', 'graph_id'], 'node': ['label', 'graph_id', 'node_id'], 'edge': ['label', 'graph_id', 'src_id', 'dst_id']}, 'num_graphs': 1, 'num_outputs': 4}


## 5. Configure and train GraphSAGE

In [5]:
pyg.SetHyperparameters(
    epochs=100,
    lr=1e-3,
    weight_decay=1e-4,
    optimizer="adamw",
    gradient_clip_norm=1.0,
    use_gpu=True,
    conv="sage",
    hidden_dims=(64, 64),
    activation="relu",
    dropout=0.20,
    batch_norm=True,
    residual=True,
)

history = pyg.Train()
print("Completed epochs:", len(history["train_loss"]))
print("Final training loss:", history["train_loss"][-1])
print("Final validation loss:", history["val_loss"][-1])


Completed epochs: 100
Final training loss: 0.05072404444217682
Final validation loss: 0.0671914666891098


## 6. Plot training history

In [6]:
figure = pyg.PlotHistory()
figure.show()


## 7. Validate and test

In [7]:
validation_metrics = pyg.Validate()
test_metrics = pyg.Test()

print("Validation metrics:")
print(json.dumps(validation_metrics, indent=2))
print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))


Validation metrics:
{
  "val_accuracy": 1.0,
  "val_precision": 1.0,
  "val_recall": 1.0,
  "val_f1": 1.0
}

Test metrics:
{
  "test_accuracy": 1.0,
  "test_precision": 1.0,
  "test_recall": 1.0,
  "test_f1": 1.0
}


## 8. Confusion matrices

In [8]:
for split in ["train", "validate", "test", "all"]:
    print("Confusion matrix:", split)
    figure = pyg.PlotConfusionMatrix(split=split)
    figure.show()


Confusion matrix: train


Confusion matrix: validate


Confusion matrix: test


Confusion matrix: all


## 9. Save the trained model

In [9]:
pyg.SaveModel(str(MODEL_PATH))
print("Saved model:", MODEL_PATH)


Saved model: C:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\assignment03\bgr_node_model.pt


## Notes

This is a small demonstration dataset. Ground and core each have only one node,
so those classes cannot appear in validation and test simultaneously. For
reliable metrics, export more buildings or create more labelled nodes for the
rare classes.
